# ССП для ИНН из Excel (февраль): OCRM vs kedr2hive

Цель:
- взять ИНН из Excel отчета за февраль;
- получить ССП двумя способами:
  1) OCRM: `ocrm_ul.s_org_ext` (+ дедуп по актуальной записи) и `cdiul.ext_id_org` для `CDI_ID`;
  2) `sandbox_ai.kedr2hive__dbo__active_clients_ul_new`;
- посчитать заполняемость, кратность ССП на 1 ИНН и количество расхождений.

In [ ]:
import re
import pandas as pd
from decimal import Decimal, InvalidOperation
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
excel_inn_col = 'ИНН'

kedr_table = 'sandbox_ai.kedr2hive__dbo__active_clients_ul_new'
kedr_inn_col = 'inn'
kedr_ssp_col = 'zo'

mem_limit = '8g'
chunk_size = 700
sample_size = 200

print('excel_path =', excel_path)
print('kedr_table =', kedr_table)
print('chunk_size =', chunk_size)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    imp.execute(f"invalidate metadata {kedr_table}")
    imp.execute('invalidate metadata ocrm_ul.s_org_ext')
    imp.execute('invalidate metadata cdiul.ext_id_org')

print('Impala initialized')

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    return s if s else None

def normalize_inn(v):
    s = normalize_numeric_str(v)
    if s is None:
        return None
    if len(s) in (10, 12):
        return s
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)
    return None

def detect_excel_header_row(path, required_cols, scan_rows=40):
    raw = pd.read_excel(path, header=None)
    req = {str(c).strip() for c in required_cols}
    for i in range(min(scan_rows, len(raw))):
        vals = {str(x).strip() for x in raw.iloc[i].tolist()}
        if len(req & vals) >= 1:
            return i
    return 0

def to_sql_in_list(values):
    out = []
    for v in values:
        s = str(v).replace("'", "''")
        out.append("'" + s + "'")
    return ', '.join(out)

## 1) ИНН из Excel

In [ ]:
header_row = detect_excel_header_row(excel_path, [excel_inn_col])
excel_raw = pd.read_excel(excel_path, header=header_row)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

if excel_inn_col not in excel_raw.columns:
    raise ValueError(f'Колонка ИНН не найдена: {excel_inn_col}')

excel_df = excel_raw[[excel_inn_col]].copy()
excel_df.columns = ['inn_raw']
excel_df['inn_norm'] = excel_df['inn_raw'].apply(normalize_inn)

excel_inn_unique = excel_df[excel_df['inn_norm'].notna()][['inn_norm']].drop_duplicates().copy()
excel_inn_list = sorted(excel_inn_unique['inn_norm'].tolist())

display(pd.DataFrame([
    {'metric': 'excel_rows_total', 'value': len(excel_df)},
    {'metric': 'excel_valid_inn_rows', 'value': int(excel_df['inn_norm'].notna().sum())},
    {'metric': 'excel_unique_inn', 'value': len(excel_inn_unique)},
    {'metric': 'excel_invalid_inn_rows', 'value': int(excel_df['inn_norm'].isna().sum())}
]))

print('inn_list_size =', len(excel_inn_list))

## 2) ССП через OCRM (`s_org_ext`) + CDI (`ext_id_org`)

In [ ]:
ocrm_chunks = []
for i in range(0, len(excel_inn_list), chunk_size):
    chunk = excel_inn_list[i:i + chunk_size]
    in_list = to_sql_in_list(chunk)

    sql_ocrm = f"""
    with t as (
      select
        row_id,
        regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '') as inn_norm,
        case
          when trim(cast(x_area_resp as string)) like 'ДММБ%'
            or trim(cast(x_area_resp as string)) like 'ДМСБ (ми%'
            then 'ДМ'
          when trim(cast(x_area_resp as string)) like 'ДМСБ (ма%'
            then 'ДМСБ'
          when trim(cast(x_area_resp as string)) like 'ДМСБ (ср%'
            or trim(cast(x_area_resp as string)) like 'ДСБ%'
            then 'ДМСБ'
          when trim(cast(x_area_resp as string)) like 'ДКБ%'
            then 'ДКБ'
          when trim(cast(x_area_resp as string)) like 'ДРПА%'
            then 'ДРПА'
          when lower(trim(cast(x_area_resp as string))) like 'не под%'
            then 'Не подлежит сегментации'
          else 'None'
        end as ssp_ocrm,
        created,
        row_number() over (
          partition by regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '')
          order by created desc, row_id desc
        ) as rn
      from ocrm_ul.s_org_ext
      where x_removed_flg = 'N'
        and x_duplicate_flg = 'N'
        and regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '') in ({in_list})
    ),
    m as (
      select distinct
        cast(party_id as string) as cdi_id,
        cast(cmo_ext_party_source_id as string) as row_id
      from cdiul.ext_id_org
      where cmo_ext_source_system like 'OCRM%'
    )
    select
      t.inn_norm,
      t.row_id,
      m.cdi_id,
      t.ssp_ocrm
    from t
    left join m on m.row_id = cast(t.row_id as string)
    where t.rn = 1
    """

    with imp:
      imp.execute(f"set MEM_LIMIT={mem_limit}")
      ocrm_chunks.append(imp.fetch(sql_ocrm))

ocrm_df = pd.concat(ocrm_chunks, ignore_index=True) if len(ocrm_chunks) else pd.DataFrame(columns=['inn_norm','row_id','cdi_id','ssp_ocrm'])
display(ocrm_df.head(sample_size))
print('ocrm_rows =', len(ocrm_df))

## 3) ССП через `kedr2hive`

In [ ]:
kedr_chunks = []
for i in range(0, len(excel_inn_list), chunk_size):
    chunk = excel_inn_list[i:i + chunk_size]
    in_list = to_sql_in_list(chunk)

    sql_kedr = f"""
    with src as (
      select
        case
          when length(regexp_replace(trim(cast({kedr_inn_col} as string)), '[^0-9]', '')) = 9
            then lpad(regexp_replace(trim(cast({kedr_inn_col} as string)), '[^0-9]', ''), 10, '0')
          when length(regexp_replace(trim(cast({kedr_inn_col} as string)), '[^0-9]', '')) = 11
            then lpad(regexp_replace(trim(cast({kedr_inn_col} as string)), '[^0-9]', ''), 12, '0')
          else regexp_replace(trim(cast({kedr_inn_col} as string)), '[^0-9]', '')
        end as inn_norm,
        case
          when upper(regexp_replace(trim(cast({kedr_ssp_col} as string)), '[^А-ЯA-Z0-9.]', '')) like 'ДМКБ%' then 'ДМ'
          when upper(regexp_replace(trim(cast({kedr_ssp_col} as string)), '[^А-ЯA-Z0-9.]', '')) like 'ДМБ%' then 'ДМСБ'
          when upper(regexp_replace(trim(cast({kedr_ssp_col} as string)), '[^А-ЯA-Z0-9.]', '')) like 'ДМСБ%' then 'ДМСБ'
          when upper(regexp_replace(trim(cast({kedr_ssp_col} as string)), '[^А-ЯA-Z0-9.]', '')) like 'ДКБ%' then 'ДКБ'
          when upper(regexp_replace(trim(cast({kedr_ssp_col} as string)), '[^А-ЯA-Z0-9.]', '')) like 'ДРА%'
            or upper(regexp_replace(trim(cast({kedr_ssp_col} as string)), '[^А-ЯA-Z0-9.]', '')) like 'ДРПА%'
            then 'ДРПА'
          when upper(regexp_replace(trim(cast({kedr_ssp_col} as string)), '[^А-ЯA-Z0-9.]', '')) like 'НЕ.ПОДЛ.СЕГМ%'
            then 'не подлежит сегментации'
          else null
        end as ssp_kedr
      from {kedr_table}
      where {kedr_inn_col} is not null
    )
    select inn_norm, ssp_kedr
    from src
    where inn_norm in ({in_list})
    """

    with imp:
      imp.execute(f"set MEM_LIMIT={mem_limit}")
      kedr_chunks.append(imp.fetch(sql_kedr))

kedr_df = pd.concat(kedr_chunks, ignore_index=True) if len(kedr_chunks) else pd.DataFrame(columns=['inn_norm','ssp_kedr'])
display(kedr_df.head(sample_size))
print('kedr_rows =', len(kedr_df))

## 4) Кратность ССП на 1 ИНН + заполняемость

In [ ]:
def ssp_status(c):
    if c == 0:
        return 'ssp_missing'
    if c == 1:
        return 'ssp_unique'
    return 'ssp_conflict'

if len(ocrm_df):
    ocrm_card = (
        ocrm_df.groupby('inn_norm', as_index=False)
        .agg(
            ocrm_rows=('inn_norm', 'count'),
            cdi_candidates=('cdi_id', lambda s: s.dropna().astype(str).nunique()),
            ssp_candidates_ocrm=('ssp_ocrm', lambda s: s.dropna().astype(str).nunique())
        )
    )
    ocrm_single = (
        ocrm_df[ocrm_df['ssp_ocrm'].notna()]
        .drop_duplicates(['inn_norm', 'ssp_ocrm'])
        .groupby('inn_norm', as_index=False)
        .agg(ssp_ocrm=('ssp_ocrm', 'min'))
    )
    ocrm_card = ocrm_card.merge(ocrm_single, on='inn_norm', how='left')
    ocrm_card['ocrm_status'] = ocrm_card['ssp_candidates_ocrm'].apply(ssp_status)
    ocrm_card['ocrm_multi_ssp_flag'] = ocrm_card['ssp_candidates_ocrm'] > 1
else:
    ocrm_card = pd.DataFrame(columns=['inn_norm','ocrm_rows','cdi_candidates','ssp_candidates_ocrm','ssp_ocrm','ocrm_status','ocrm_multi_ssp_flag'])

if len(kedr_df):
    kedr_card = (
        kedr_df.groupby('inn_norm', as_index=False)
        .agg(
            kedr_rows=('inn_norm', 'count'),
            ssp_candidates_kedr=('ssp_kedr', lambda s: s.dropna().astype(str).nunique())
        )
    )
    kedr_single = (
        kedr_df[kedr_df['ssp_kedr'].notna()]
        .drop_duplicates(['inn_norm', 'ssp_kedr'])
        .groupby('inn_norm', as_index=False)
        .agg(ssp_kedr=('ssp_kedr', 'min'))
    )
    kedr_card = kedr_card.merge(kedr_single, on='inn_norm', how='left')
    kedr_card['kedr_status'] = kedr_card['ssp_candidates_kedr'].apply(ssp_status)
else:
    kedr_card = pd.DataFrame(columns=['inn_norm','kedr_rows','ssp_candidates_kedr','ssp_kedr','kedr_status'])

display(ocrm_card.head(sample_size))
display(kedr_card.head(sample_size))

## 5) Сравнение OCRM vs kedr2hive и количество расхождений

In [ ]:
cmp_df = excel_inn_unique.copy()
cmp_df = cmp_df.merge(ocrm_card, on='inn_norm', how='left')
cmp_df = cmp_df.merge(kedr_card, on='inn_norm', how='left')

for c in ['ssp_candidates_ocrm', 'ssp_candidates_kedr', 'cdi_candidates']:
    if c in cmp_df.columns:
        cmp_df[c] = pd.to_numeric(cmp_df[c], errors='coerce').fillna(0).astype(int)

cmp_df['ocrm_status'] = cmp_df['ssp_candidates_ocrm'].apply(ssp_status)
cmp_df['kedr_status'] = cmp_df['ssp_candidates_kedr'].apply(ssp_status)

def discrepancy_type(r):
    o_cnt = r['ssp_candidates_ocrm']
    k_cnt = r['ssp_candidates_kedr']
    o_val = None if pd.isna(r.get('ssp_ocrm')) else str(r.get('ssp_ocrm'))
    k_val = None if pd.isna(r.get('ssp_kedr')) else str(r.get('ssp_kedr'))

    if o_cnt == 0 and k_cnt == 0:
        return 'both_missing'
    if o_cnt > 0 and k_cnt == 0:
        return 'only_ocrm'
    if o_cnt == 0 and k_cnt > 0:
        return 'only_kedr'
    if o_cnt == 1 and k_cnt == 1:
        return 'both_unique_match' if o_val == k_val else 'both_unique_mismatch'
    if o_cnt != k_cnt:
        return 'different_multiplicity'
    return 'both_multi_or_conflict'

cmp_df['discrepancy_type'] = cmp_df.apply(discrepancy_type, axis=1)

discrepancy_summary = (
    cmp_df.groupby('discrepancy_type', as_index=False)
    .agg(inn_cnt=('inn_norm', 'count'))
    .sort_values('inn_cnt', ascending=False)
)
discrepancy_summary['share'] = discrepancy_summary['inn_cnt'] / max(len(cmp_df), 1)

total_discrepancies = int((cmp_df['discrepancy_type'] != 'both_unique_match').sum())

inn_ocrm_multi_ssp_after_filters = int((cmp_df['ssp_candidates_ocrm'] > 1).sum())

kpi = pd.DataFrame([
    {'metric': 'excel_unique_inn', 'value': len(cmp_df)},
    {'metric': 'ocrm_ssp_unique', 'value': int((cmp_df['ocrm_status'] == 'ssp_unique').sum())},
    {'metric': 'kedr_ssp_unique', 'value': int((cmp_df['kedr_status'] == 'ssp_unique').sum())},
    {'metric': 'total_discrepancies', 'value': total_discrepancies},
    {'metric': 'both_unique_mismatch', 'value': int((cmp_df['discrepancy_type'] == 'both_unique_mismatch').sum())},
    {'metric': 'only_ocrm', 'value': int((cmp_df['discrepancy_type'] == 'only_ocrm').sum())},
    {'metric': 'only_kedr', 'value': int((cmp_df['discrepancy_type'] == 'only_kedr').sum())},
    {'metric': 'inn_ocrm_multi_ssp_after_filters', 'value': inn_ocrm_multi_ssp_after_filters},
])
kpi['share'] = kpi['value'] / max(len(cmp_df), 1)

display(kpi)
display(discrepancy_summary)
display(cmp_df.head(sample_size))

print('total_discrepancies =', total_discrepancies)
print('inn_ocrm_multi_ssp_after_filters =', inn_ocrm_multi_ssp_after_filters)

## 6) Примеры расхождений

Рекомендуется смотреть `both_unique_mismatch`, `only_ocrm`, `only_kedr` в первую очередь.

## 7) ИНН с несколькими ССП в OCRM после всех фильтров

Метрика после фильтров тетрадки (`x_removed_flg='N'`, `x_duplicate_flg='N'`, периметр ИНН Excel, нормализация ССП):
- `ssp_candidates_ocrm > 1`.

In [ ]:
ocrm_multi_ssp_df = cmp_df[cmp_df['ssp_candidates_ocrm'] > 1][[
    'inn_norm', 'cdi_candidates', 'ssp_candidates_ocrm', 'ssp_ocrm', 'ocrm_status'
]].copy()

display(ocrm_multi_ssp_df.head(sample_size))
print('inn_ocrm_multi_ssp_after_filters =', len(ocrm_multi_ssp_df))
print('share_ocrm_multi_ssp_after_filters =', round(len(ocrm_multi_ssp_df) / max(len(cmp_df), 1), 6))

In [ ]:
important_types = ['both_unique_mismatch', 'only_ocrm', 'only_kedr', 'different_multiplicity']
samples = cmp_df[cmp_df['discrepancy_type'].isin(important_types)][[
    'inn_norm', 'cdi_candidates', 'ssp_candidates_ocrm', 'ssp_ocrm', 'ocrm_status',
    'ssp_candidates_kedr', 'ssp_kedr', 'kedr_status', 'discrepancy_type'
]].copy()
display(samples.head(sample_size))
print('important_discrepancy_rows =', len(samples))